# 📡 Notebook 01 — Multi-Source Data Collection
## Spatio-Temporal PM2.5/PM10 Analysis — Karachi, Pakistan

**Sources:**
- `COPERNICUS/S5P/NRTI/L3_AER_AI` — Sentinel-5P Aerosol Index (UV-based PM proxy)
- `COPERNICUS/S5P/NRTI/L3_NO2` — Sentinel-5P Nitrogen Dioxide (combustion proxy)
- `COPERNICUS/S5P/NRTI/L3_SO2` — Sentinel-5P Sulphur Dioxide (industrial proxy)
- `COPERNICUS/S5P/NRTI/L3_CO` — Sentinel-5P Carbon Monoxide (traffic proxy)
- `MODIS/061/MOD08_D3` — MODIS Terra Aerosol Optical Depth (daily global)
- `ECMWF/ERA5_LAND/DAILY_AGGR` — ERA5 Meteorological Reanalysis
- `NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG` — VIIRS Nighttime Light (activity proxy)
- `COPERNICUS/S2_SR_HARMONIZED` — Sentinel-2 NDVI (green cover)
- Kaggle Global Air Pollution Dataset (ground-truth PM values)

**Output:** `data/raw/` — individual CSVs per source + `data/processed/merged_karachi_dataset.csv`

---
> ⚠️ **Run this notebook once on good internet.** All GEE exports go to Google Drive automatically.
> After this notebook completes, all subsequent work is 100% offline.

## 0. Imports & Configuration

In [ ]:
import ee
import geemap
import pandas as pd
import numpy as np
import os
import json
import time
import warnings
from datetime import datetime, timedelta
from pathlib import Path
import requests
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

warnings.filterwarnings('ignore')

# ── Project Configuration ─────────────────────────────────────────────────────
GEE_PROJECT   = 'gen-lang-client-0478151371'
START_DATE    = '2019-01-01'
END_DATE      = '2024-01-01'
DRIVE_FOLDER  = 'karachi_airq_exports'   # GDrive folder for all exports

# Karachi bounding box (generous — covers full city + surroundings)
KARACHI_BBOX = {
    'lon_min': 66.60, 'lon_max': 67.60,
    'lat_min': 24.60, 'lat_max': 25.20
}

# Karachi city center
KARACHI_CENTER = [67.0011, 24.8607]

# Key monitoring stations (lat, lon, name)
STATIONS = [
    (24.9056, 67.0822, 'Gulshan-e-Iqbal'),
    (24.8560, 67.0100, 'Saddar'),
    (24.9400, 66.9800, 'SITE_Industrial'),
    (24.8200, 67.0300, 'Korangi_Industrial'),
    (24.9800, 67.1200, 'North_Nazimabad'),
    (24.8900, 67.1300, 'Gulistan_Jauhar'),
    (24.8100, 66.9900, 'Landhi'),
    (24.9200, 67.0500, 'Federal_B_Area'),
]

# Create project directories
for d in ['data/raw', 'data/processed', 'data/spatial', 'outputs', 'models']:
    Path(d).mkdir(parents=True, exist_ok=True)

print('✓ Configuration loaded')
print(f'  Study period : {START_DATE} → {END_DATE}')
print(f'  Stations     : {len(STATIONS)} monitoring points across Karachi')
print(f'  Drive folder : {DRIVE_FOLDER}')

## 1. Initialize Google Earth Engine

In [ ]:
ee.Initialize(project=GEE_PROJECT)

# Define Karachi geometry objects for GEE
karachi_point  = ee.Geometry.Point(KARACHI_CENTER)
karachi_rect   = ee.Geometry.Rectangle([
    KARACHI_BBOX['lon_min'], KARACHI_BBOX['lat_min'],
    KARACHI_BBOX['lon_max'], KARACHI_BBOX['lat_max']
])

# Station feature collection for spatial sampling
station_features = ee.FeatureCollection([
    ee.Feature(ee.Geometry.Point([lon, lat]), {'station': name, 'lat': lat, 'lon': lon})
    for lat, lon, name in STATIONS
])

print('✓ GEE initialized')
print(f'  Project      : {GEE_PROJECT}')
print(f'  Karachi bbox : {KARACHI_BBOX}')

# Quick sanity check — print dataset sizes
print('\n📊 Dataset inventory:')
datasets = {
    'S5P Aerosol'  : 'COPERNICUS/S5P/NRTI/L3_AER_AI',
    'S5P NO2'      : 'COPERNICUS/S5P/NRTI/L3_NO2',
    'S5P SO2'      : 'COPERNICUS/S5P/NRTI/L3_SO2',
    'S5P CO'       : 'COPERNICUS/S5P/NRTI/L3_CO',
    'ERA5 Daily'   : 'ECMWF/ERA5_LAND/DAILY_AGGR',
}
for name, asset_id in datasets.items():
    try:
        n = ee.ImageCollection(asset_id) \
            .filterDate(START_DATE, END_DATE) \
            .filterBounds(karachi_rect) \
            .size().getInfo()
        print(f'  {name:<18}: {n:>5} images')
    except Exception as ex:
        print(f'  {name:<18}: ERROR - {ex}')

## 2. Helper Functions

In [ ]:
def extract_timeseries_at_stations(collection, band_names, reducer=ee.Reducer.mean(), scale=1000):
    """
    Extract spatially-averaged values from an ImageCollection at each station point.
    Returns a pandas DataFrame with columns: date, station, lat, lon, band_values...
    
    Uses reduceRegions for efficient batch processing — much faster than mapping
    over images individually.
    """
    def extract_image(image):
        date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')
        reduced = image.select(band_names).reduceRegions(
            collection=station_features,
            reducer=reducer,
            scale=scale
        )
        return reduced.map(lambda f: f.set('date', date))
    
    results = collection.map(extract_image).flatten()
    return results


def gee_fc_to_dataframe(fc, max_features=50000):
    """
    Convert GEE FeatureCollection to pandas DataFrame.
    Handles large collections by batching getInfo calls.
    """
    size = fc.size().getInfo()
    print(f'  Converting {size} features to DataFrame...')
    
    all_features = []
    batch_size = 5000
    
    for start in tqdm(range(0, min(size, max_features), batch_size), desc='Batching'):
        batch = fc.toList(batch_size, start)
        features = ee.FeatureCollection(batch).getInfo()['features']
        for feat in features:
            props = feat['properties']
            all_features.append(props)
        time.sleep(0.2)  # polite rate limiting
    
    return pd.DataFrame(all_features)


def export_to_drive(fc, description, folder=DRIVE_FOLDER):
    """
    Queue a FeatureCollection export to Google Drive.
    Non-blocking — returns task object.
    """
    task = ee.batch.Export.table.toDrive(
        collection=fc,
        description=description,
        folder=folder,
        fileFormat='CSV'
    )
    task.start()
    print(f'  ✓ Export queued: {description} → Google Drive/{folder}/')
    return task


def monitor_tasks(tasks, poll_interval=30):
    """
    Monitor GEE export tasks until completion.
    Prints status updates every poll_interval seconds.
    """
    print('\n⏳ Monitoring export tasks...')
    while True:
        statuses = [t.status() for t in tasks]
        states   = [s['state'] for s in statuses]
        
        for i, (task, status) in enumerate(zip(tasks, statuses)):
            state = status['state']
            desc  = status['description']
            print(f'  [{i+1}] {desc:<40} → {state}')
        
        if all(s in ['COMPLETED', 'FAILED', 'CANCELLED'] for s in states):
            completed = sum(1 for s in states if s == 'COMPLETED')
            failed    = sum(1 for s in states if s == 'FAILED')
            print(f'\n✓ All tasks finished: {completed} completed, {failed} failed')
            break
        
        print(f'  (checking again in {poll_interval}s...)\n')
        time.sleep(poll_interval)


print('✓ Helper functions loaded')

## 3. Sentinel-5P — Aerosol Index, NO2, SO2, CO

Sentinel-5P TROPOMI provides daily near-global coverage at 3.5km×5.5km resolution.
We extract four pollutant channels simultaneously at all 8 Karachi stations.

In [ ]:
print('📡 Extracting Sentinel-5P multi-pollutant data...')

# ── S5P Aerosol Index ─────────────────────────────────────────────────────────
s5p_aer = (
    ee.ImageCollection('COPERNICUS/S5P/NRTI/L3_AER_AI')
    .filterDate(START_DATE, END_DATE)
    .filterBounds(karachi_rect)
    .select(['absorbing_aerosol_index'])
)

# ── S5P NO2 ───────────────────────────────────────────────────────────────────
s5p_no2 = (
    ee.ImageCollection('COPERNICUS/S5P/NRTI/L3_NO2')
    .filterDate(START_DATE, END_DATE)
    .filterBounds(karachi_rect)
    .select(['NO2_column_number_density'])
)

# ── S5P SO2 ───────────────────────────────────────────────────────────────────
s5p_so2 = (
    ee.ImageCollection('COPERNICUS/S5P/NRTI/L3_SO2')
    .filterDate(START_DATE, END_DATE)
    .filterBounds(karachi_rect)
    .select(['SO2_column_number_density'])
)

# ── S5P CO ────────────────────────────────────────────────────────────────────
s5p_co = (
    ee.ImageCollection('COPERNICUS/S5P/NRTI/L3_CO')
    .filterDate(START_DATE, END_DATE)
    .filterBounds(karachi_rect)
    .select(['CO_column_number_density'])
)

print(f'  AER_AI images : {s5p_aer.size().getInfo()}')
print(f'  NO2 images    : {s5p_no2.size().getInfo()}')
print(f'  SO2 images    : {s5p_so2.size().getInfo()}')
print(f'  CO images     : {s5p_co.size().getInfo()}')

In [ ]:
print('⚙️  Extracting S5P time series at station points...')
print('   (This may take 3-5 minutes — all computation is on Google servers)')

tasks_s5p = []

# Extract + export each pollutant
for collection, band, label in [
    (s5p_aer, 'absorbing_aerosol_index',       'aer_ai'),
    (s5p_no2, 'NO2_column_number_density',      'no2'),
    (s5p_so2, 'SO2_column_number_density',      'so2'),
    (s5p_co,  'CO_column_number_density',       'co'),
]:
    fc = extract_timeseries_at_stations(collection, [band], scale=5000)
    task = export_to_drive(fc, f'karachi_s5p_{label}_{START_DATE[:4]}_{END_DATE[:4]}')
    tasks_s5p.append(task)

print(f'\n✓ {len(tasks_s5p)} S5P export tasks queued to Google Drive')

## 4. MODIS — Aerosol Optical Depth (AOD)

MOD08_D3 provides daily Level-3 global atmosphere products at 1-degree resolution.
AOD at 550nm is the standard satellite proxy for surface PM2.5 used in published literature.
We use the `Aerosol_Optical_Depth_Land_Mean` band.

In [ ]:
print('🛰️  Extracting MODIS AOD data...')

modis_aod = (
    ee.ImageCollection('MODIS/061/MOD08_D3')
    .filterDate(START_DATE, END_DATE)
    .filterBounds(karachi_rect)
    .select([
        'Aerosol_Optical_Depth_Land_Mean',
        'Aerosol_Optical_Depth_Land_Ocean_Mean',
        'Aerosol_Optical_Depth_Land_Std_Mean',
    ])
)

# Apply MODIS scale factor (AOD values need ×0.001)
def apply_modis_scale(image):
    scaled = image.multiply(0.001).copyProperties(image, ['system:time_start'])
    return scaled

modis_aod_scaled = modis_aod.map(apply_modis_scale)

print(f'  MODIS AOD images (global daily): {modis_aod.size().getInfo()}')
print('  Note: Resolution is 1-degree (~111km) — appropriate for city-level analysis')

# Extract at Karachi center point (1-degree resolution — one value covers whole city)
def extract_modis_point(image):
    date  = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')
    vals  = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=karachi_rect,
        scale=111000,
        maxPixels=1e9
    )
    return ee.Feature(None, vals.set('date', date).set('location', 'Karachi'))

modis_fc = modis_aod_scaled.map(extract_modis_point)
task_modis = export_to_drive(modis_fc, f'karachi_modis_aod_{START_DATE[:4]}_{END_DATE[:4]}')

print('\n✓ MODIS AOD export queued')

## 5. ERA5 — Meteorological Reanalysis

ERA5-Land daily aggregates from ECMWF. Meteorological variables are critical features
for PM prediction: boundary layer height controls vertical mixing, wind disperses pollution,
humidity affects particle formation, temperature drives photochemical reactions.

In [ ]:
print('🌤️  Extracting ERA5 meteorological reanalysis...')

ERA5_BANDS = [
    'temperature_2m',                    # Surface temperature (K)
    'temperature_2m_min',                # Min daily temp
    'temperature_2m_max',                # Max daily temp
    'dewpoint_temperature_2m',           # Dew point → relative humidity proxy
    'u_component_of_wind_10m',           # East-West wind component
    'v_component_of_wind_10m',           # North-South wind component
    'surface_pressure',                  # Surface pressure (Pa)
    'total_precipitation_sum',           # Precipitation (wet deposition)
    'evaporation_from_bare_soil_sum',    # Soil dryness proxy (dust resuspension)
]

era5 = (
    ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
    .filterDate(START_DATE, END_DATE)
    .filterBounds(karachi_rect)
    .select(ERA5_BANDS)
)

print(f'  ERA5 images: {era5.size().getInfo()} daily records')

# Derive wind speed and direction as engineered features
def add_wind_features(image):
    u = image.select('u_component_of_wind_10m')
    v = image.select('v_component_of_wind_10m')
    wind_speed = u.pow(2).add(v.pow(2)).sqrt().rename('wind_speed_10m')
    wind_dir   = u.atan2(v).multiply(180 / 3.14159).rename('wind_direction_deg')
    # Relative humidity from temperature and dewpoint (Magnus formula approximation)
    t  = image.select('temperature_2m').subtract(273.15)       # K → °C
    td = image.select('dewpoint_temperature_2m').subtract(273.15)
    rh = td.subtract(t).multiply(100/23.0).add(100).rename('relative_humidity')
    return image.addBands([wind_speed, wind_dir, rh])

era5_enriched = era5.map(add_wind_features)

# Extract at Karachi bbox
def extract_era5(image):
    date = ee.Date(image.get('system:time_start')).format('YYYY-MM-dd')
    vals = image.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=karachi_rect,
        scale=9000,
        maxPixels=1e9
    )
    return ee.Feature(None, vals.set('date', date))

era5_fc = era5_enriched.map(extract_era5)
task_era5 = export_to_drive(era5_fc, f'karachi_era5_meteo_{START_DATE[:4]}_{END_DATE[:4]}')

print('\n✓ ERA5 export queued')
print(f'  Bands extracted: {ERA5_BANDS + ["wind_speed_10m", "wind_direction_deg", "relative_humidity"]}')

## 6. VIIRS — Nighttime Light (Activity Proxy)

VIIRS DNB monthly composites serve as a proxy for anthropogenic activity.
Higher nighttime radiance correlates with vehicular traffic and industrial activity.
Used as a static spatial feature engineering input.

In [ ]:
print('🌃 Extracting VIIRS Nighttime Light data...')

viirs_ntl = (
    ee.ImageCollection('NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG')
    .filterDate(START_DATE, END_DATE)
    .filterBounds(karachi_rect)
    .select(['avg_rad'])   # Average radiance (nW/cm²/sr)
)

print(f'  VIIRS NTL images: {viirs_ntl.size().getInfo()} monthly composites')

# Extract per-station monthly NTL values
def extract_viirs(image):
    date   = ee.Date(image.get('system:time_start')).format('YYYY-MM')
    reduced = image.reduceRegions(
        collection=station_features,
        reducer=ee.Reducer.mean(),
        scale=500
    )
    return reduced.map(lambda f: f.set('year_month', date))

viirs_fc   = viirs_ntl.map(extract_viirs).flatten()
task_viirs = export_to_drive(viirs_fc, f'karachi_viirs_ntl_{START_DATE[:4]}_{END_DATE[:4]}')

print('\n✓ VIIRS NTL export queued')

## 7. Sentinel-2 — NDVI (Green Cover Index)

Normalized Difference Vegetation Index from Sentinel-2 SR.
Monthly median composites with cloud masking (SCL band).
Lower NDVI areas (industrial, paved) tend to accumulate more PM.

In [ ]:
print('🌿 Extracting Sentinel-2 NDVI (monthly composites)...')

def mask_s2_clouds(image):
    """Mask clouds using Sentinel-2 Scene Classification Layer (SCL)."""
    scl = image.select('SCL')
    # Keep: vegetation(4), bare soil(5), water(6), unclassified(7), medium prob cloud(8 excluded)
    mask = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10)).And(scl.neq(11))
    return image.updateMask(mask)

def add_ndvi(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return image.addBands(ndvi)

def add_ndbi(image):
    """Normalized Difference Built-up Index — higher in urban/industrial areas."""
    ndbi = image.normalizedDifference(['B11', 'B8']).rename('NDBI')
    return image.addBands(ndbi)

# Monthly composites to reduce cloud gaps
months = ee.List.sequence(0, 59)  # 60 months = 5 years
start  = ee.Date(START_DATE)

def monthly_composite(m):
    m       = ee.Number(m)
    start_m = start.advance(m, 'month')
    end_m   = start_m.advance(1, 'month')
    composite = (
        ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
        .filterDate(start_m, end_m)
        .filterBounds(karachi_rect)
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
        .map(mask_s2_clouds)
        .map(add_ndvi)
        .map(add_ndbi)
        .select(['NDVI', 'NDBI'])
        .median()
        .set('year_month', start_m.format('YYYY-MM'))
        .set('system:time_start', start_m.millis())
    )
    return composite

s2_monthly  = ee.ImageCollection(months.map(monthly_composite))

# Extract NDVI + NDBI per station
def extract_s2(image):
    ym      = image.get('year_month')
    reduced = image.reduceRegions(
        collection=station_features,
        reducer=ee.Reducer.mean(),
        scale=20
    )
    return reduced.map(lambda f: f.set('year_month', ym))

s2_fc      = s2_monthly.map(extract_s2).flatten()
task_s2    = export_to_drive(s2_fc, f'karachi_s2_ndvi_ndbi_{START_DATE[:4]}_{END_DATE[:4]}')

print('\n✓ Sentinel-2 NDVI/NDBI export queued')
print('  Features: NDVI (green cover), NDBI (built-up intensity)')

## 8. Load Kaggle Ground-Truth PM Dataset

In [ ]:
print('📊 Loading Kaggle ground-truth PM dataset...')

# ── Try to load Kaggle dataset ─────────────────────────────────────────────────
KAGGLE_PATH = 'data/raw/global_air_pollution_dataset.csv'

if not Path(KAGGLE_PATH).exists():
    print(f'  ⚠️  File not found at {KAGGLE_PATH}')
    print('  → Download from: https://www.kaggle.com/datasets/hasibalmuzdadid/global-air-pollution-dataset')
    print('  → Place CSV in data/raw/ folder')
    print('  → Re-run this cell')
    kaggle_df = None
else:
    df_raw = pd.read_csv(KAGGLE_PATH)
    print(f'  Full dataset shape: {df_raw.shape}')
    print(f'  Columns: {list(df_raw.columns)}')
    
    # Filter to Pakistan / Karachi
    country_col = [c for c in df_raw.columns if 'country' in c.lower()]
    city_col    = [c for c in df_raw.columns if 'city' in c.lower()]
    
    if country_col:
        pk_df = df_raw[df_raw[country_col[0]].str.contains('Pakistan', na=False, case=False)]
        print(f'\n  Pakistan records: {len(pk_df)}')
        if city_col:
            print(f'  Cities: {pk_df[city_col[0]].unique()}')
        karachi_df = pk_df[pk_df[city_col[0]].str.contains('Karachi', na=False, case=False)] if city_col else pk_df
        print(f'  Karachi records: {len(karachi_df)}')
        karachi_df.to_csv('data/raw/karachi_kaggle_pm.csv', index=False)
        print('\n  ✓ Karachi subset saved → data/raw/karachi_kaggle_pm.csv')
        kaggle_df = karachi_df
    else:
        print('  Could not auto-filter — check column names above and filter manually')
        kaggle_df = df_raw
    
    print(f'\n  Sample:')
    display(kaggle_df.head())

## 9. Monitor All GEE Export Tasks

In [ ]:
all_tasks = tasks_s5p + [task_modis, task_era5, task_viirs, task_s2]

print(f'📤 Total GEE export tasks queued: {len(all_tasks)}')
print(f'   Destination: Google Drive / {DRIVE_FOLDER}/')
print()

# Print current status
for i, task in enumerate(all_tasks):
    status = task.status()
    print(f'  [{i+1}] {status["description"]:<55} → {status["state"]}')

print()
print('💡 TIP: You can close this notebook now.')
print('   GEE processes exports on Google servers — no need to keep this running.')
print('   Check task status at: https://code.earthengine.google.com/tasks')
print('   Files will appear in Google Drive once done (usually 5-20 minutes).')

In [ ]:
# OPTIONAL: Run this cell only if you want to wait and monitor live
# Otherwise skip — exports continue on Google's servers automatically

# Uncomment to monitor:
# monitor_tasks(all_tasks, poll_interval=30)

## 10. Load Exported CSVs from Drive & Build Master Dataset

⚠️ **Run this section AFTER all GEE exports complete.**
Download all CSVs from Google Drive into `data/raw/` then run cells below.

In [ ]:
def load_gee_csv(pattern, label):
    """Load a GEE-exported CSV from data/raw/."""
    matches = list(Path('data/raw').glob(f'*{pattern}*.csv'))
    if not matches:
        print(f'  ⚠️  No file matching *{pattern}*.csv in data/raw/')
        return None
    df = pd.read_csv(matches[0])
    # Parse date column
    date_col = [c for c in df.columns if 'date' in c.lower()]
    if date_col:
        df[date_col[0]] = pd.to_datetime(df[date_col[0]])
        df = df.rename(columns={date_col[0]: 'date'})
    print(f'  ✓ {label:<25}: {df.shape[0]:>6} rows, {df.shape[1]} cols → {matches[0].name}')
    return df

print('📂 Loading GEE exports from data/raw/...')
df_aer   = load_gee_csv('aer_ai',  'S5P Aerosol Index')
df_no2   = load_gee_csv('no2',     'S5P NO2')
df_so2   = load_gee_csv('so2',     'S5P SO2')
df_co    = load_gee_csv('co',      'S5P CO')
df_modis = load_gee_csv('modis',   'MODIS AOD')
df_era5  = load_gee_csv('era5',    'ERA5 Meteorology')
df_viirs = load_gee_csv('viirs',   'VIIRS Nighttime Light')
df_s2    = load_gee_csv('ndvi',    'Sentinel-2 NDVI/NDBI')

In [ ]:
print('🔗 Building master merged dataset...')

# ── Step 1: Merge all station-level daily S5P data ────────────────────────────
station_dfs = []
for df, col_rename in [
    (df_aer,  {'absorbing_aerosol_index': 'aer_ai'}),
    (df_no2,  {'NO2_column_number_density': 'no2'}),
    (df_so2,  {'SO2_column_number_density': 'so2'}),
    (df_co,   {'CO_column_number_density': 'co'}),
]:
    if df is not None:
        df = df.rename(columns=col_rename)
        station_dfs.append(df[['date', 'station', 'lat', 'lon'] + list(col_rename.values())])

if station_dfs:
    from functools import reduce
    merged = reduce(lambda l, r: pd.merge(l, r, on=['date','station','lat','lon'], how='outer'), station_dfs)
    print(f'  S5P merged shape: {merged.shape}')
else:
    print('  ⚠️  No S5P data loaded yet — complete GEE exports first')
    merged = pd.DataFrame()

# ── Step 2: Merge ERA5 (date-level, broadcast to all stations) ─────────────────
if df_era5 is not None and not merged.empty:
    # ERA5 is city-wide (no station column) — merge on date only
    era5_clean = df_era5.drop(columns=[c for c in df_era5.columns if c in ['station','lat','lon']], errors='ignore')
    merged = pd.merge(merged, era5_clean, on='date', how='left')
    print(f'  After ERA5 merge: {merged.shape}')

# ── Step 3: Merge MODIS AOD (date-level) ──────────────────────────────────────
if df_modis is not None and not merged.empty:
    modis_clean = df_modis[['date', 'Aerosol_Optical_Depth_Land_Mean',
                             'Aerosol_Optical_Depth_Land_Ocean_Mean']].copy()
    modis_clean.columns = ['date', 'modis_aod_land', 'modis_aod_ocean']
    merged = pd.merge(merged, modis_clean, on='date', how='left')
    print(f'  After MODIS merge: {merged.shape}')

# ── Step 4: Merge VIIRS NTL (monthly — forward-fill to daily) ─────────────────
if df_viirs is not None and not merged.empty:
    df_viirs['year_month'] = df_viirs['year_month'].astype(str).str[:7]
    merged['year_month']   = merged['date'].dt.strftime('%Y-%m')
    merged = pd.merge(merged, df_viirs[['year_month','station','avg_rad']].rename(
        columns={'avg_rad':'viirs_ntl'}), on=['year_month','station'], how='left')
    merged.drop(columns='year_month', inplace=True)
    print(f'  After VIIRS merge: {merged.shape}')

# ── Step 5: Merge S2 NDVI (monthly per station) ───────────────────────────────
if df_s2 is not None and not merged.empty:
    df_s2['year_month']  = df_s2['year_month'].astype(str).str[:7]
    merged['year_month'] = merged['date'].dt.strftime('%Y-%m')
    merged = pd.merge(merged, df_s2[['year_month','station','NDVI','NDBI']].rename(
        columns={'NDVI':'ndvi','NDBI':'ndbi'}), on=['year_month','station'], how='left')
    merged.drop(columns='year_month', inplace=True)
    print(f'  After NDVI merge: {merged.shape}')

if not merged.empty:
    print(f'\n✅ Master dataset shape: {merged.shape}')
    print(f'   Date range: {merged["date"].min()} → {merged["date"].max()}')
    print(f'   Stations  : {merged["station"].nunique()}')
    print(f'   Features  : {list(merged.columns)}')

## 11. Feature Engineering — Temporal Features

In [ ]:
if not merged.empty:
    print('⚙️  Engineering temporal and domain features...')
    df = merged.copy().sort_values(['station', 'date'])

    # ── Calendar features ──────────────────────────────────────────────────────
    df['year']        = df['date'].dt.year
    df['month']       = df['date'].dt.month
    df['day_of_year'] = df['date'].dt.dayofyear
    df['day_of_week'] = df['date'].dt.dayofweek
    df['is_weekend']  = (df['day_of_week'] >= 5).astype(int)
    df['quarter']     = df['date'].dt.quarter

    # ── Season encoding (Karachi climate) ─────────────────────────────────────
    # Winter: Nov-Feb (high PM, temp inversion), Summer: Mar-Jun, Monsoon: Jul-Oct
    def season(month):
        if month in [11, 12, 1, 2]:  return 'winter'
        elif month in [3, 4, 5, 6]:  return 'summer'
        else:                         return 'monsoon'
    df['season']       = df['month'].map(season)
    df['is_winter']    = (df['season'] == 'winter').astype(int)
    df['is_monsoon']   = (df['season'] == 'monsoon').astype(int)

    # ── Cyclical encoding (sin/cos — for ML models) ────────────────────────────
    df['month_sin']   = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']   = np.cos(2 * np.pi * df['month'] / 12)
    df['dow_sin']     = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['dow_cos']     = np.cos(2 * np.pi * df['day_of_week'] / 7)

    # ── Pakistan-specific event flags ─────────────────────────────────────────
    # Eid dates (approximate — major traffic/fireworks spikes)
    EID_DATES = [
        '2019-06-04', '2019-08-11', '2020-05-24', '2020-07-31',
        '2021-05-13', '2021-07-20', '2022-05-02', '2022-07-09',
        '2023-04-21', '2023-06-28',
    ]
    eid_dates_parsed = pd.to_datetime(EID_DATES)
    # Flag ±3 days around each Eid
    df['is_eid_period'] = 0
    for eid in eid_dates_parsed:
        mask = (df['date'] >= eid - pd.Timedelta(days=3)) & \
               (df['date'] <= eid + pd.Timedelta(days=3))
        df.loc[mask, 'is_eid_period'] = 1

    # Ramadan flag (approximate)
    RAMADAN_RANGES = [
        ('2019-05-06', '2019-06-03'), ('2020-04-24', '2020-05-23'),
        ('2021-04-13', '2021-05-12'), ('2022-04-02', '2022-05-01'),
        ('2023-03-23', '2023-04-20'),
    ]
    df['is_ramadan'] = 0
    for start_r, end_r in RAMADAN_RANGES:
        mask = (df['date'] >= start_r) & (df['date'] <= end_r)
        df.loc[mask, 'is_ramadan'] = 1

    # ── Lag features (per station) ─────────────────────────────────────────────
    for col in ['aer_ai', 'no2', 'so2', 'co', 'modis_aod_land']:
        if col in df.columns:
            for lag in [1, 3, 7, 14]:
                df[f'{col}_lag{lag}'] = df.groupby('station')[col].shift(lag)

    # ── Rolling statistics (per station) ──────────────────────────────────────
    for col in ['aer_ai', 'no2', 'modis_aod_land']:
        if col in df.columns:
            df[f'{col}_roll7_mean']  = df.groupby('station')[col].transform(lambda x: x.rolling(7,  min_periods=1).mean())
            df[f'{col}_roll30_mean'] = df.groupby('station')[col].transform(lambda x: x.rolling(30, min_periods=1).mean())
            df[f'{col}_roll7_std']   = df.groupby('station')[col].transform(lambda x: x.rolling(7,  min_periods=1).std())

    print(f'  ✓ Feature engineering complete')
    print(f'  Final shape: {df.shape}')
    print(f'  New features added: calendar, season, Eid/Ramadan flags, {4} lag sets, rolling stats')

    # Save processed dataset
    df.to_csv('data/processed/merged_karachi_dataset.csv', index=False)
    print(f'\n  ✅ Saved → data/processed/merged_karachi_dataset.csv')

## 12. Quick Data Quality Report

In [ ]:
if not merged.empty:
    print('📋 DATA QUALITY REPORT')
    print('=' * 60)
    
    print(f'\nShape         : {df.shape}')
    print(f'Date range    : {df["date"].min().date()} → {df["date"].max().date()}')
    print(f'Stations      : {df["station"].nunique()} ({list(df["station"].unique())})')
    print(f'Total days    : {df["date"].nunique()}')
    
    print('\nMissing values per key feature:')
    key_cols = ['aer_ai', 'no2', 'so2', 'co', 'modis_aod_land',
                'temperature_2m', 'wind_speed_10m', 'relative_humidity',
                'total_precipitation_sum', 'viirs_ntl', 'ndvi']
    for col in key_cols:
        if col in df.columns:
            missing_pct = df[col].isna().mean() * 100
            bar = '█' * int(missing_pct / 5) + '░' * (20 - int(missing_pct / 5))
            print(f'  {col:<35} {bar} {missing_pct:.1f}%')
    
    print('\nDescriptive statistics (key pollutants):')
    stat_cols = [c for c in ['aer_ai','no2','so2','co','modis_aod_land'] if c in df.columns]
    if stat_cols:
        display(df[stat_cols].describe().round(4))

## 13. Quick Visualization — Time Series Preview

In [ ]:
if not merged.empty and 'aer_ai' in df.columns:
    fig, axes = plt.subplots(3, 1, figsize=(16, 12), facecolor='#0d0d14')
    
    COLORS = ['#c8f04a', '#4af0c8', '#f04a7a', '#f0c84a', '#7a4af0',
              '#4a7af0', '#f07a4a', '#4af07a']
    
    for ax in axes:
        ax.set_facecolor('#111118')
        ax.tick_params(colors='#888')
        for spine in ax.spines.values():
            spine.set_edgecolor('#222')
    
    # Plot 1: Aerosol Index per station
    ax = axes[0]
    for i, (station, grp) in enumerate(df.groupby('station')):
        daily = grp.groupby('date')['aer_ai'].mean()
        ax.plot(daily.index, daily.values, alpha=0.7, linewidth=0.8,
                color=COLORS[i % len(COLORS)], label=station)
    ax.set_title('Sentinel-5P Absorbing Aerosol Index — Karachi Stations (2019–2024)',
                 color='#e8e8f0', fontsize=11, pad=10)
    ax.set_ylabel('AAI', color='#888')
    ax.legend(fontsize=7, ncol=4, facecolor='#16161f', labelcolor='#aaa',
              edgecolor='#222', loc='upper right')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', color='#666')
    
    # Plot 2: NO2 city average
    if 'no2' in df.columns:
        ax = axes[1]
        city_avg = df.groupby('date')['no2'].mean().rolling(7).mean()
        ax.fill_between(city_avg.index, city_avg.values, alpha=0.3, color='#4af0c8')
        ax.plot(city_avg.index, city_avg.values, color='#4af0c8', linewidth=1)
        ax.set_title('NO₂ Column Density — Karachi City Average (7-day rolling mean)',
                     color='#e8e8f0', fontsize=11, pad=10)
        ax.set_ylabel('mol/m²', color='#888')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', color='#666')
    
    # Plot 3: MODIS AOD
    if 'modis_aod_land' in df.columns:
        ax = axes[2]
        aod = df.groupby('date')['modis_aod_land'].mean().rolling(14).mean()
        ax.fill_between(aod.index, aod.values, alpha=0.3, color='#f0c84a')
        ax.plot(aod.index, aod.values, color='#f0c84a', linewidth=1)
        ax.set_title('MODIS Aerosol Optical Depth (550nm) — Karachi (14-day rolling mean)',
                     color='#e8e8f0', fontsize=11, pad=10)
        ax.set_ylabel('AOD', color='#888')
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
        plt.setp(ax.xaxis.get_majorticklabels(), rotation=45, ha='right', color='#666')
    
    plt.suptitle('Karachi Air Quality — Raw Data Preview', color='#fff',
                 fontsize=14, fontweight='bold', y=1.01)
    plt.tight_layout()
    plt.savefig('outputs/01_data_preview.png', dpi=150, bbox_inches='tight',
                facecolor='#0d0d14', edgecolor='none')
    plt.show()
    print('\n✓ Preview saved → outputs/01_data_preview.png')

## 14. Summary & Next Steps

In [ ]:
print('=' * 60)
print('✅ NOTEBOOK 01 COMPLETE — DATA COLLECTION')
print('=' * 60)
print()
print('GEE Exports queued (check Google Drive):')
print('  karachi_s5p_aer_ai_2019_2024.csv')
print('  karachi_s5p_no2_2019_2024.csv')
print('  karachi_s5p_so2_2019_2024.csv')
print('  karachi_s5p_co_2019_2024.csv')
print('  karachi_modis_aod_2019_2024.csv')
print('  karachi_era5_meteo_2019_2024.csv')
print('  karachi_viirs_ntl_2019_2024.csv')
print('  karachi_s2_ndvi_ndbi_2019_2024.csv')
print()
print('Local files:')
print('  data/raw/karachi_kaggle_pm.csv')
print('  data/processed/merged_karachi_dataset.csv  ← MAIN DATASET')
print()
print('NEXT: Run 02_eda.ipynb')
print('  → Temporal decomposition')
print('  → Spatial station maps')
print('  → Correlation heatmaps')
print('  → Seasonal + event analysis (Eid, Ramadan, Monsoon)')